# 第 11 章: 評価指標と交差検証の探索と可視化

Survived の決定木について、混同行列・ROC 曲線・交差検証の分割ごとのスコアを確認する。

In [ ]:
%use dataframe(0.15.0), kandy(0.8.0)

In [ ]:
@file:DependsOn("org.tribuo:tribuo-classification-tree:4.3.2")

In [ ]:
@file:DependsOn("../build/libs/getting-started-ml.jar")

In [ ]:
import chapter02.splitTrainTest
import chapter03.toTribuoDataset
import chapter03.trainTribuoTree
import chapter11.DecisionTreeModel
import chapter11.N_SPLITS
import chapter11.SEED
import chapter11.SURVIVED_METRICS
import chapter11.confusionMatrix
import chapter11.crossValidate
import chapter11.f1Score
import chapter11.kFold
import chapter11.precision
import chapter11.prepareSurvived
import chapter11.recall
import org.tribuo.classification.evaluation.LabelEvaluationUtil
import java.io.File

// Notebook は notebooks/ で実行されるので、学習データの既定の場所を 1 つ上にずらす
val survivedCsv = File(dataset.dataDir { name -> System.getenv(name) ?: "../../data/sukkiri-ml" }, "Survived.csv")
val (x, t) = prepareSurvived(DataFrame.readCSV(survivedCsv))

## 混同行列

In [ ]:
val split = splitTrainTest(x, t, testSize = 0.3, seed = 0)
val model = DecisionTreeModel(maxDepth = 2)
model.fit(split.xTrain, split.tTrain)
val predicted = model.predict(split.xTest)
val cm = confusionMatrix(split.tTest, predicted, positive = "1")
cm

In [ ]:
val cells =
    dataFrameOf(
        "予測" to listOf("死亡と予測", "生存と予測", "死亡と予測", "生存と予測"),
        "実際" to listOf("実際は死亡", "実際は死亡", "実際は生存", "実際は生存"),
        "件数" to listOf(cm.tn, cm.fp, cm.fn, cm.tp),
    )
cells.plot {
    tiles {
        x("予測")
        y("実際")
        fillColor("件数")
    }
    text {
        x("予測")
        y("実際")
        label("件数")
    }
    layout.title = "混同行列（テストデータ）"
}

In [ ]:
mapOf("適合率" to precision(cm), "再現率" to recall(cm), "F値" to f1Score(cm))

## ROC 曲線（Tribuo の予測スコア）

第 3 章の自作の決定木はラベルしか返さないので、ROC 曲線は「生存」の確率を返す Tribuo の決定木で描く。

In [ ]:
val testDataset = toTribuoDataset(split.xTest, split.tTest)

fun survivalScores(maxDepth: Int): DoubleArray {
    val tree = trainTribuoTree(split.xTrain, split.tTrain, maxDepth, minChildWeight = 1.0f)
    return tree.predict(testDataset).map { it.outputScores.getValue("1").score }.toDoubleArray()
}

val isSurvived = split.tTest.map { it == "1" }.toBooleanArray()
val roc = LabelEvaluationUtil.generateROCCurve(isSurvived, survivalScores(maxDepth = 2))
dataFrameOf(
    "偽陽性率" to roc.fpr.toList(),
    "真陽性率" to roc.tpr.toList(),
).plot {
    line {
        x("偽陽性率")
        y("真陽性率")
    }
    layout.title = "ROC 曲線（Tribuo の決定木、深さ 2）"
}

In [ ]:
listOf(1, 2, 4, 8).associateWith { depth -> LabelEvaluationUtil.binaryAUCROC(isSurvived, survivalScores(depth)) }

## 交差検証の分割ごとのスコア

In [ ]:
val folds = kFold(nSamples = x.rowsCount(), nSplits = N_SPLITS, seed = SEED)
val foldScores =
    SURVIVED_METRICS.map { (name, metric) ->
        name to crossValidate({ DecisionTreeModel(maxDepth = 2) }, x, t, folds, metric).toList()
    }
val scores =
    dataFrameOf(
        "分割" to foldScores.flatMap { (_, values) -> values.indices.map { it + 1 } },
        "指標" to foldScores.flatMap { (name, values) -> values.map { name } },
        "スコア" to foldScores.flatMap { (_, values) -> values },
    )
scores.plot {
    points {
        x("分割")
        y("スコア")
        color("指標")
    }
    layout.title = "分割ごとのスコア（決定木、深さ 2）"
}

In [ ]:
scores.groupBy("指標").aggregate {
    min("スコア") into "最小"
    max("スコア") into "最大"
}